# Refinamiento BTS

In [1]:
from pyspark.sql import SparkSession, functions as F, types as T

spark = (SparkSession.builder.appName("refinamiento_bts")
         .config("spark.driver.memory","2g").enableHiveSupport().getOrCreate())
spark.conf.set("spark.sql.shuffle.partitions","64")

RAW_BTS = "/Obligatorio/landing/bts"
REFINED = "/Obligatorio/refined"

bts_raw = (spark.read.option("header",True).option("recursiveFileLookup",True)
           .option("pathGlobFilter","*.csv").option("quote",'"').option("escape",'"')
           .option("mode","PERMISSIVE").csv(RAW_BTS))
# quitar la columna fantasma del header con coma final
fantasma = [c for c in bts_raw.columns if c.strip()=="" or c.startswith("_c")]
if fantasma: bts_raw = bts_raw.drop(*fantasma)

def lim(c):
    s = F.trim(F.col(c).cast("string"))
    return F.when(s.isin(["","\\N"]), None).otherwise(s)

flights = (
    bts_raw.select(
        F.col("Year").cast("int").alias("year"),
        F.col("Month").cast("int").alias("month"),
        F.to_date(lim("FlightDate")).alias("flight_date"),
        F.upper(lim("Reporting_Airline")).alias("reporting_airline"),
        lim("Flight_Number_Reporting_Airline").alias("flight_number"),
        F.upper(lim("Origin")).alias("origin"),
        F.upper(lim("Dest")).alias("dest"),
        F.col("CRSDepTime").cast("int").alias("crs_dep_time"),
        F.col("DepDelay").cast("double").alias("dep_delay"),
        F.col("DepDel15").cast("double").alias("dep_del15"),
        F.col("ArrDelay").cast("double").alias("arr_delay"),
        F.col("ArrDel15").cast("double").alias("arr_del15"),
        F.col("Cancelled").cast("double").alias("cancelled"),
        F.col("Diverted").cast("double").alias("diverted"),
    )
    .filter(F.col("year").isNotNull() & F.col("month").isNotNull() & F.col("flight_date").isNotNull() &
            F.col("origin").isNotNull() & F.col("dest").isNotNull() & F.col("reporting_airline").isNotNull())
    .filter(F.col("year") >= 2023)
    # quita los duplicados (las 53.885 filas exactas son un subconjunto de esta identidad de vuelo)
    .dropDuplicates(["flight_date","reporting_airline","flight_number","origin","dest","crs_dep_time"])
)
print("Vuelos refinados:", flights.count())



Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


2026-06-24T00:44:31,508 WARN [Thread-4] org.apache.hadoop.util.NativeCodeLoader - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


[Stage 4:===================================================>       (7 + 1) / 8]

Vuelos refinados: 20928579


In [2]:
# Calendario amplio (cubre BTS y también las fechas de eventos de MusicBrainz)
dim_date = (
    spark.sql("SELECT explode(sequence(to_date('1950-01-01'), to_date('2035-12-31'), interval 1 day)) AS full_date")
    .withColumn("date_id", F.date_format("full_date","yyyyMMdd").cast("int"))
    .withColumn("year", F.year("full_date"))
    .withColumn("month", F.month("full_date"))
    .withColumn("day", F.dayofmonth("full_date"))
    .withColumn("week_of_year", F.weekofyear("full_date"))
    .withColumn("quarter", F.quarter("full_date"))
    .select("date_id","full_date","year","month","day","week_of_year","quarter")
)
print("dim_date:", dim_date.count())
dim_date.write.mode("overwrite").parquet(f"{REFINED}/dim_date")

dim_date: 31411


In [3]:
# Salidas (por origen) y llegadas (por destino)
dep = (flights.groupBy(F.col("origin").alias("airport_iata"),"flight_date","year","month")
       .agg(F.count("*").alias("departures"),
            F.avg("dep_delay").alias("avg_dep_delay"),
            F.sum(F.when(F.col("dep_del15")==1,1).otherwise(0)).alias("dep_delayed_15"),
            F.sum(F.when(F.col("cancelled")==1,1).otherwise(0)).alias("cancelled_flights"),
            F.sum(F.when(F.col("diverted")==1,1).otherwise(0)).alias("diverted_flights")))
arr = (flights.groupBy(F.col("dest").alias("airport_iata"),"flight_date","year","month")
       .agg(F.count("*").alias("arrivals"),
            F.avg("arr_delay").alias("avg_arr_delay"),
            F.sum(F.when(F.col("arr_del15")==1,1).otherwise(0)).alias("arr_delayed_15")))

daily = (dep.join(arr, ["airport_iata","flight_date","year","month"], "full")
    .fillna({"departures":0,"arrivals":0,"dep_delayed_15":0,"arr_delayed_15":0,
             "cancelled_flights":0,"diverted_flights":0})
    .withColumn("total_flights", F.col("departures")+F.col("arrivals"))
    .withColumn("total_delayed_15", F.col("dep_delayed_15")+F.col("arr_delayed_15"))
    .withColumn("delay_15_rate", F.when(F.col("total_flights")>0, F.col("total_delayed_15")/F.col("total_flights")).otherwise(0.0))
    .withColumn("cancellation_rate", F.when(F.col("total_flights")>0, F.col("cancelled_flights")/F.col("total_flights")).otherwise(0.0))
).cache()
print("airport-día:", daily.count())

# Mapa IATA -> airport_id (de dim_airport, OpenFlights)
iata_id = (spark.read.parquet(f"{REFINED}/dim_airport")
           .select("iata_code","airport_id").filter(F.col("iata_code").isNotNull())
           .dropDuplicates(["iata_code"]))

no_match = daily.select("airport_iata").distinct().join(
    iata_id.withColumnRenamed("iata_code","airport_iata"), "airport_iata","left_anti").count()
print("IATA de BTS sin match en dim_airport:", no_match)

fact_airport_daily_ops = (
    daily.join(iata_id.withColumnRenamed("iata_code","airport_iata"), "airport_iata","inner")
    .withColumn("date_id", F.date_format("flight_date","yyyyMMdd").cast("int"))
    .select("airport_id","date_id","departures","arrivals","total_flights",
            "avg_dep_delay","avg_arr_delay","cancelled_flights","diverted_flights",
            "delay_15_rate","cancellation_rate","year")
)
fact_airport_daily_ops.write.mode("overwrite").partitionBy("year").parquet(f"{REFINED}/fact_airport_daily_ops")
print("fact_airport_daily_ops escrita")

airport-día: 360344


IATA de BTS sin match en dim_airport: 1


fact_airport_daily_ops escrita


In [4]:
weekly = (
    daily.withColumn("week_of_year", F.weekofyear("flight_date"))
    .groupBy("airport_iata","year","week_of_year")
    .agg(F.min("flight_date").alias("week_start_date"),
         F.max("flight_date").alias("week_end_date"),
         F.sum("total_flights").alias("total_flights"),
         F.avg("avg_dep_delay").alias("avg_dep_delay"),
         F.avg("avg_arr_delay").alias("avg_arr_delay"),
         F.sum("cancelled_flights").alias("cancelled_flights"),
         F.sum("total_delayed_15").alias("total_delayed_15"))
    .withColumn("weekly_cancellation_rate", F.when(F.col("total_flights")>0, F.col("cancelled_flights")/F.col("total_flights")).otherwise(0.0))
    .withColumn("weekly_delay_15_rate", F.when(F.col("total_flights")>0, F.col("total_delayed_15")/F.col("total_flights")).otherwise(0.0))
)
fact_airport_weekly_ops = (
    weekly.join(iata_id.withColumnRenamed("iata_code","airport_iata"), "airport_iata","inner")
    .select("airport_id","week_of_year","week_start_date","week_end_date","total_flights",
            "avg_dep_delay","avg_arr_delay","weekly_cancellation_rate","weekly_delay_15_rate","year")
)
fact_airport_weekly_ops.write.mode("overwrite").partitionBy("year").parquet(f"{REFINED}/fact_airport_weekly_ops")
print("fact_airport_weekly_ops escrita")

[Stage 66:>                                                         (0 + 2) / 2]

fact_airport_weekly_ops escrita


In [5]:
baseline = (
    daily.groupBy("airport_iata","year")
    .agg(F.avg("total_flights").alias("baseline_avg_daily_flights"),
         F.expr("percentile_approx(total_flights, 0.5)").alias("baseline_median_daily_flights"),
         F.avg("avg_dep_delay").alias("baseline_avg_dep_delay"),
         F.avg("avg_arr_delay").alias("baseline_avg_arr_delay"),
         F.avg("cancellation_rate").alias("baseline_avg_cancellation_rate"),
         F.avg("delay_15_rate").alias("baseline_avg_delay_15_rate"))
)
fact_airport_yearly_baseline = (
    baseline.join(iata_id.withColumnRenamed("iata_code","airport_iata"), "airport_iata","inner")
    .select("airport_id","year","baseline_avg_daily_flights","baseline_median_daily_flights",
            "baseline_avg_dep_delay","baseline_avg_arr_delay",
            "baseline_avg_cancellation_rate","baseline_avg_delay_15_rate")
)
fact_airport_yearly_baseline.write.mode("overwrite").parquet(f"{REFINED}/fact_airport_yearly_baseline")
print("fact_airport_yearly_baseline:", fact_airport_yearly_baseline.count())
daily.unpersist()

fact_airport_yearly_baseline: 1047


DataFrame[airport_iata: string, flight_date: date, year: int, month: int, departures: bigint, avg_dep_delay: double, dep_delayed_15: bigint, cancelled_flights: bigint, diverted_flights: bigint, arrivals: bigint, avg_arr_delay: double, arr_delayed_15: bigint, total_flights: bigint, total_delayed_15: bigint, delay_15_rate: double, cancellation_rate: double]

In [6]:
# Las tablas particionadas externas necesitan registrar sus particiones en el metastore.
spark.sql("MSCK REPAIR TABLE festivales_aereo.fact_airport_daily_ops")
spark.sql("MSCK REPAIR TABLE festivales_aereo.fact_airport_weekly_ops")

for t in ["dim_date","fact_airport_daily_ops","fact_airport_weekly_ops","fact_airport_yearly_baseline"]:
    df = spark.read.parquet(f"{REFINED}/{t}")
    print(f"{t:30} filas={df.count():>8}")

print("\nParticiones registradas:")
spark.sql("SHOW PARTITIONS festivales_aereo.fact_airport_daily_ops").show()

2026-06-24T01:02:45,994 INFO [Thread-4] org.apache.hadoop.hive.conf.HiveConf - Found configuration file file:/home/ort/spark/conf/hive-site.xml
2026-06-24T01:02:46,275 WARN [Thread-4] org.apache.hadoop.hive.conf.HiveConf - HiveConf of name hive.metastore.wm.default.pool.size does not exist
2026-06-24T01:02:46,276 WARN [Thread-4] org.apache.hadoop.hive.conf.HiveConf - HiveConf of name hive.llap.task.scheduler.preempt.independent does not exist
2026-06-24T01:02:46,276 WARN [Thread-4] org.apache.hadoop.hive.conf.HiveConf - HiveConf of name hive.llap.output.format.arrow does not exist
2026-06-24T01:02:46,276 WARN [Thread-4] org.apache.hadoop.hive.conf.HiveConf - HiveConf of name hive.tez.llap.min.reducer.per.executor does not exist
2026-06-24T01:02:46,276 WARN [Thread-4] org.apache.hadoop.hive.conf.HiveConf - HiveConf of name hive.arrow.root.allocator.limit does not exist
2026-06-24T01:02:46,276 WARN [Thread-4] org.apache.hadoop.hive.conf.HiveConf - HiveConf of name hive.vectorized.use.che

fact_airport_daily_ops         filas=  359248
fact_airport_weekly_ops        filas=   52352
fact_airport_yearly_baseline   filas=    1047

Particiones registradas:
+---------+
|partition|
+---------+
|year=2023|
|year=2024|
|year=2025|
+---------+

